# Test Chunk To Page Mapping

This notebook tests the page-mapping logic on a single document before running it across the whole archive.

In [15]:
from pathlib import Path
import importlib
import sqlite3
import sys

PIPELINE_DIR = Path.cwd()
if PIPELINE_DIR.name != "data_pipeline":
    PIPELINE_DIR = PIPELINE_DIR / "data_pipeline"

PROJECT_ROOT = PIPELINE_DIR.parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import map_chunks_to_pages
importlib.reload(map_chunks_to_pages)
extract_page_texts = map_chunks_to_pages.extract_page_texts
best_page = map_chunks_to_pages.best_page

ARCHIVE_DB = PROJECT_ROOT / "output" / "metadata" / "archive.db"
CHUNKS_DB = PROJECT_ROOT / "output" / "metadata" / "chunks.db"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARCHIVE_DB exists:", ARCHIVE_DB.exists())
print("CHUNKS_DB exists:", CHUNKS_DB.exists())


PROJECT_ROOT: /Users/rf50/uchicago/maroon/scrape-maroon-archives
ARCHIVE_DB exists: True
CHUNKS_DB exists: True


In [16]:
# Change this doc_id to test another issue.
DOC_ID = "mvol-0004-1944-0324"
DOC_ID


'mvol-0004-1944-0324'

In [17]:
archive_conn = sqlite3.connect(ARCHIVE_DB)
archive_conn.row_factory = sqlite3.Row
chunks_conn = sqlite3.connect(CHUNKS_DB)
chunks_conn.row_factory = sqlite3.Row

doc_row = archive_conn.execute(
    "SELECT doc_id, pdf_path, text_path FROM documents WHERE doc_id = ?",
    (DOC_ID,),
).fetchone()

chunk_rows = chunks_conn.execute(
    "SELECT chunk_id, chunk_index, text FROM chunks WHERE doc_id = ? ORDER BY chunk_index LIMIT 10",
    (DOC_ID,),
).fetchall()

archive_conn.close()
chunks_conn.close()

print(dict(doc_row) if doc_row else "document not found")
print("chunk sample count:", len(chunk_rows))


{'doc_id': 'mvol-0004-1944-0324', 'pdf_path': '/Users/rf50/uchicago/maroon/scrape-maroon-archives/output/pdfs/1944/03/mvol-0004-1944-0324.pdf', 'text_path': '/Users/rf50/uchicago/maroon/scrape-maroon-archives/output/plain_text/1944/03/mvol-0004-1944-0324.txt'}
chunk sample count: 10


In [18]:
pdf_path = Path(doc_row["pdf_path"])
page_texts = extract_page_texts(pdf_path)
print("num pages:", len(page_texts))

for i, page_text in enumerate(page_texts[:5], start=1):
    print("=" * 80)
    print("PAGE", i)
    print(page_text[:600])


num pages: 6
PAGE 1
Chicago Maroon
VoI.3,No.23 Z-149 Friday, March 24, 1944 Price Five Cents
President Robert M. Hutchins
Confers Degrees Today To One
Hundred Sixty-one Graduates
President Robert Maynard Hutch¬
ins will confer degrees on one hundred
sixty-one graduates this afternoon at
three o’clock in Rockefeller Memorial
Chapel in the University’s two hun¬
dred sixteenth convocation ceremony.
Sixty-three degrees will be con¬
ferred in absentia, thirty-four of them
to Army Air Force meteorology sec¬
ond lieutenants now on active‘duty.
Eighty-six candidates for the bach¬
elor’s degree in arts, science, and phil¬
PAGE 2
Page Two
Music Critic Deplores Faction
Element In Chicago "Culture"
Chicago, the second largest city in
the land, presents the anomaly of
heading for oblivion as a musical cen¬
ter. (Query: Was it ever one?) If you
ask, “What has this to do with the
University of Chicago?”, the answer
is that one of the largest single areas
of support of musical activity in this
town is

In [19]:
results = []
for row in chunk_rows:
    page_number, score = best_page(row["text"], page_texts)
    results.append(
        {
            "chunk_id": row["chunk_id"],
            "chunk_index": row["chunk_index"],
            "page_number": page_number,
            "page_match_score": score,
            "chunk_preview": row["text"][:400],
        }
    )

results


[{'chunk_id': 'mvol-0004-1944-0324::chunk-0000',
  'chunk_index': 0,
  'page_number': 1,
  'page_match_score': 0.9157894736842105,
  'chunk_preview': 'Chicago Maroon Vo I.3, No.23 Z-149 Friday, March 24, 1944 Price Five Cents President Robert M. Hutchins Confers Degrees Today To One Hundred Sixty-one Graduates President Robert Maynard Hutchins will confer degrees on one hundred sixty-one graduates this afternoon at three oclock in Rockefeller Memorial Chapel in the Universitys two hundred sixteenth convocation ceremony. Sixty-three degrees will '},
 {'chunk_id': 'mvol-0004-1944-0324::chunk-0001',
  'chunk_index': 1,
  'page_number': 1,
  'page_match_score': 0.9081885856079405,
  'chunk_preview': 'Th^ convocation address will be delivered by Chester Whitney Wright, Professor of Economics. His topic is“The Growing Responsibilities of the Citizen.” Following the conferring of degrees and awarding of commissions. President Hutchins will make the convocation statement, a quarterly announcem

In [20]:
for item in results:
    print("=" * 100)
    print(item["chunk_id"])
    print("chunk index:", item["chunk_index"])
    print("mapped page:", item["page_number"])
    print("score:", round(item["page_match_score"], 4))
    print(item["chunk_preview"])


mvol-0004-1944-0324::chunk-0000
chunk index: 0
mapped page: 1
score: 0.9158
Chicago Maroon Vo I.3, No.23 Z-149 Friday, March 24, 1944 Price Five Cents President Robert M. Hutchins Confers Degrees Today To One Hundred Sixty-one Graduates President Robert Maynard Hutchins will confer degrees on one hundred sixty-one graduates this afternoon at three oclock in Rockefeller Memorial Chapel in the Universitys two hundred sixteenth convocation ceremony. Sixty-three degrees will 
mvol-0004-1944-0324::chunk-0001
chunk index: 1
mapped page: 1
score: 0.9082
Th^ convocation address will be delivered by Chester Whitney Wright, Professor of Economics. His topic is“The Growing Responsibilities of the Citizen.” Following the conferring of degrees and awarding of commissions. President Hutchins will make the convocation statement, a quarterly announcement of gifts made to the University, appointments made to, the faculty, and, sometimes, information concer
mvol-0004-1944-0324::chunk-0002
chunk index: 2

In [ ]:
# After running map_chunks_to_pages.py, inspect what was actually written to chunks.db.\nstored_conn = sqlite3.connect(CHUNKS_DB)\nstored_conn.row_factory = sqlite3.Row\n\nstored_rows = stored_conn.execute(\n    \"\"\"\n    SELECT chunk_id, chunk_index, page_number, page_match_score\n    FROM chunks\n    WHERE doc_id = ?\n    ORDER BY chunk_index\n    LIMIT 10\n    \"\"\",\n    (DOC_ID,),\n).fetchall()\n\nstored_conn.close()\nstored_rows\n

In [ ]:
for predicted, stored in zip(results, stored_rows):\n    print(\"=\" * 100)\n    print(predicted[\"chunk_id\"])\n    print(\n        \"predicted page:\",\n        predicted[\"page_number\"],\n        \"stored page:\",\n        stored[\"page_number\"],\n    )\n    print(\n        \"predicted score:\",\n        round(predicted[\"page_match_score\"], 4),\n        \"stored score:\",\n        None if stored[\"page_match_score\"] is None else round(stored[\"page_match_score\"], 4),\n    )\n